# XGBoost + KMeansSMOTE Model

This notebook is a reproducible runner for the `XGBoost + KMeansSMOTE` model.

It keeps the main implementation inside the corresponding Python script in `src/`,
while this notebook provides a clean, reviewable execution workflow for the project repository.

## Expected project structure

Run this notebook from the `notebooks/` folder or from the project root.

Required local files:

```text
Spaceship-Titanic-MLW/
├── data/
│   ├── train.csv
│   ├── test.csv
│   └── sample_submission.csv   # needed by some models
├── src/
├── submissions/
└── notebooks/
```

The Kaggle data files do not need to be uploaded to GitHub.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root. Please run this notebook inside the project folder.")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
RUN_DIR = PROJECT_ROOT / "_notebook_run"

SUBMISSION_DIR.mkdir(exist_ok=True)
RUN_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Source folder:", SRC_DIR)
print("Data folder:", DATA_DIR)
print("Submission folder:", SUBMISSION_DIR)

In [ ]:
script_candidates = ['train_xgboost_kmeanssmote.py', 'train_XGBoost + KMeansSMOTE.py', 'XGBoost + KMeansSMOTE.py', 'xgb_kmeans_pipeline.py']

SCRIPT_PATH = None
for name in script_candidates:
    candidate = SRC_DIR / name
    if candidate.exists():
        SCRIPT_PATH = candidate
        break

if SCRIPT_PATH is None:
    raise FileNotFoundError(
        "Could not find the model script. Checked: " + ", ".join(str(SRC_DIR / name) for name in script_candidates)
    )

print("Using script:", SCRIPT_PATH)

In [ ]:
required_data = ["train.csv", "test.csv"]
missing = [name for name in required_data if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing required data files in {DATA_DIR}: {missing}")

print("Data files are available.")

## Run the model script

In [ ]:
work_dir = RUN_DIR / "xgboost_kmeanssmote"
work_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(DATA_DIR / "train.csv", work_dir / "train.csv")
shutil.copy2(DATA_DIR / "test.csv", work_dir / "test.csv")
if not (DATA_DIR / "sample_submission.csv").exists():
    raise FileNotFoundError("sample_submission.csv is required for the XGBoost + KMeansSMOTE script.")
shutil.copy2(DATA_DIR / "sample_submission.csv", work_dir / "sample_submission.csv")

output_dir = SUBMISSION_DIR / "xgboost_kmeanssmote_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--output-dir", str(output_dir),
]

print("Running:", " ".join(str(x) for x in cmd))
completed = subprocess.run(cmd, cwd=work_dir, text=True, capture_output=True)

print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(f"Model script failed with return code {completed.returncode}")

print("Outputs saved to:", output_dir)

## Notes

- The model implementation is stored in `src/`.
- This notebook is intended for review, reproduction, and demonstration.
- If the script name is later standardized, this notebook already checks multiple candidate names.